# 会说话的主播：一张角色图 + 一段文稿 → 口播视频

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dawang-sakura/colab-talking-avatar/blob/main/talking_avatar.ipynb)

```
文稿 ──VoxCPM2 配音──▶ 语音 wav ──ComfyUI + InfiniteTalk 对嘴──▶ 无声视频 ──合成＋烧字幕──▶ talking_avatar.mp4
```

全程在 Google Colab 的 **L4 GPU** 上运行，你的电脑不用安装任何东西。

**使用步骤**
1. 菜单「代码执行程序 → 更改运行时类型」（英文界面：Runtime → Change runtime type）选 **L4 GPU** → 保存。
2. 只改下面的 **① 设置** 格（不改也能直接跑示例）。
3. 「代码执行程序 → 全部运行」（英文界面：Runtime → Run all）。弹出「此笔记本不是由 Google 创作」时点「仍然运行」。
4. 跑完浏览器会自动下载 `talking_avatar.mp4`（默认带字幕），另附字幕文件 `talking_avatar.srt`。
5. **用完务必「代码执行程序 → 断开连接并删除运行时」**（英文界面：Runtime → Disconnect and delete runtime），闲置也会扣计算单元。

> **使用规范**：只能使用你本人、已取得当事人同意、或 AI 生成的虚构角色的脸和声音；禁止冒充他人。Colab 明文禁止 creating deepfakes（[Colab FAQ](https://research.google.com/colaboratory/faq.html)）。

没用过 Colab？请看仓库里的《新手操作手册》：`docs/新手操作手册.md`。

In [ ]:
# ① 设置（只需要改这一格）

# ===== 文稿 =====
# 一行一句；每句最多 80 字，超过会自动切开（VoxCPM2 单次太长容易出现啸叫或音色漂移）
# 句首可以用括号写语气，例如：(轻松地)大家好。开启「极致克隆」（见下方）时不能加语气括号
SCRIPT = """
大家好，我是虚构的 AI 主播。
一张图加一段文稿，就能让我开口说话。
全程在 Colab 上运行，不用装任何东西。
"""

# ===== 声音 =====
# 声音来源的优先顺序：
#   1. 开启了下方「声音克隆」→ 用你的录音
#   2. 否则 VOICE 选了声音包里的声音 → 用仓库 voices/ 里的参考音（试听见 README 的「声音包」）
#   3. VOICE 选「自定义（用 VOICE_DESC）」→ 照 VOICE_DESC 的文字描述做音色设计
VOICE = "新闻女主播"  #@param ["新闻女主播", "温柔知性", "元气少女", "甜美邻家", "成熟御姐", "热情带货", "沉稳旁白", "阳光青年", "亲切大叔", "激情解说", "自定义（用 VOICE_DESC）"]
VOICE_DESC = "年轻女性，声音清亮自然，语速中等，语气亲切平稳，像新闻主播在播报"   # 音色设计用的文字描述（VOICE 选「自定义」时才用）
PAUSE_SEC = 0.4      # 句与句之间的停顿（秒）
SEED = 42            # 随机种子：音色设计、逐句配音、对嘴生成都用它；对声音不满意可以换一个数字重跑
                     # 用声音包时，音色由参考音固定，SEED 只影响每句的细微差异和对嘴画面

# ===== 角色图 =====
# 默认用仓库自带的示例图（AI 生成的虚构人物）
# 换成自己的图：上传到左侧「文件」面板（会放在 /content/），再改成 "/content/你的图.png"
IMAGE_PATH = "examples/host.png"
# 对嘴画面的英文描述，可按角色外观补充（例如发型、服装）
PROMPT = ("A person is talking to the camera, facing the camera, upper body, "
          "natural head movement, clear mouth movement.")

# ===== 字幕（默认开启）=====
# 字幕文字直接用上面的文稿原文，时间按每句配音的实际长度推算（不做语音识别，所以不会有错字）
# 一句太长、一行放不下时会自动切成几段轮流显示，每段时间按字数比例分配
# 成品会把字幕烧进画面，另外再给一份 talking_avatar.srt
# 跑完想换字幕样式：改这一段 → 运行 ① → 运行「⑥ 只重做字幕」格，不用重跑配音和对嘴
SUBTITLE = True  #@param {type:"boolean"}
# 字体：都是开源、可商用的字体，运行时从官方发布处下载（授权见 README「字幕」）
SUB_FONT = "思源黑体（粗）"  #@param ["思源黑体（粗）", "霞鹜文楷", "站酷快乐体", "站酷小薇LOGO体", "站酷庆科黄油体", "自定义（上传字体文件）"]
# SUB_FONT 选「自定义」时才用：把 .ttf／.otf 上传到左侧「文件」面板（/content/）后填路径；请自行确认该字体允许你的用途（例如商用）
SUB_FONT_FILE = "/content/my_font.ttf"
# 字号（%）：100＝标准大小（画面短边的 1/12，例如 480×832 画面约 40 像素）；字越大，一行放的字越少
SUB_SIZE = 100  #@param {type:"slider", min:50, max:200, step:5}
# 出现效果：淡入＝每段字幕淡淡出现；淡入淡出＝消失时也淡出；弹出＝放大弹出；滑入＝从下往上滑进来；逐字＝按念的速度一个字一个字出现
SUB_EFFECT = "淡入"  #@param ["无", "淡入", "淡入淡出", "弹出", "滑入", "逐字"]
SUB_POSITION = "底部"  #@param ["底部", "中间", "顶部"]
# 字幕离画面下边缘（选「顶部」时是上边缘）的距离，占画面高度的 %；选「中间」时不用
SUB_MARGIN = 12  #@param {type:"slider", min:0, max:45, step:1}
# 描边粗细，占字号的 %；0＝不描边（默认单层黑色描边）
SUB_OUTLINE = 8  #@param {type:"slider", min:0, max:20, step:1}
# 颜色写成 #RRGGBB，例如白色 #FFFFFF、黄色 #FFE000、黑色 #000000
SUB_COLOR = "#FFFFFF"  #@param {type:"string"}
SUB_OUTLINE_COLOR = "#000000"  #@param {type:"string"}

# ===== 显存 =====
LOW_VRAM = False     # 对嘴阶段爆显存（OutOfMemory）时改成 True 再重跑：更慢但更省显存

# ===== 声音克隆（默认关闭）=====
# 只能用你本人、或已取得当事人同意的录音。开启后 VOICE 和 VOICE_DESC 都不再使用
ENABLE_VOICE_CLONE = False
REF_WAV = "/content/my_voice.wav"   # 参考录音（上传到 /content/）；建议 5～15 秒、单人说话、没有背景音乐
REF_TEXT = ""                       # 参考录音的逐字稿：填了走「极致克隆」（更像本人，但不能加语气括号）；留空走「可控克隆」
I_HAVE_CONSENT = False              # 确认录音是你本人、或已取得当事人同意后，才改成 True

if ENABLE_VOICE_CLONE and I_HAVE_CONSENT is not True:
    raise PermissionError("已开启声音克隆，但 I_HAVE_CONSENT 不是 True。"
                          "只能用你本人或已取得当事人同意的录音；确认后把 I_HAVE_CONSENT 改成 True 再运行。")
print("设置已读取")

In [ ]:
# ② 环境检查：GPU、示例文件、输入文件
import os, subprocess, sys, time

T_START = time.time()
REPO_URL = "https://github.com/dawang-sakura/colab-talking-avatar"
REPO_DIR = "/content/colab-talking-avatar"
OUT_DIR = "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)


def run(cmd, **kw):
    """运行外部命令；失败时打印最后 40 行输出并报错。"""
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        tail = "\n".join((r.stdout + r.stderr).splitlines()[-40:])
        raise RuntimeError(f"命令失败（exit {r.returncode}）：{' '.join(map(str, cmd))}\n{tail}")
    return r


def gpu_used():
    return subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()


# GPU：本项目只在 L4（约 22.5GB 显存）上实测过
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
if not smi:
    raise RuntimeError("没有检测到 GPU：请到「代码执行程序 → 更改运行时类型」选 L4 GPU，保存后重新全部运行")
GPU_NAME, GPU_MIB = [s.strip() for s in smi.splitlines()[0].rsplit(",", 1)]
GPU_MIB = int(GPU_MIB)
print(f"GPU：{GPU_NAME}，显存 {GPU_MIB / 1024:.1f} GiB")
if "L4" not in GPU_NAME or GPU_MIB < 20 * 1024:
    print("⚠ 警告：当前不是 L4，或显存不足 20GB。本项目只在 L4 上实测过，T4（15GB）未测试，"
          "很可能爆显存或非常慢。\n"
          "  请到「代码执行程序 → 更改运行时类型」改成 L4；"
          "浏览器登录了多个 Google 账号时，Colab 默认用第一个账号，该账号没有计算单元会被降成 T4。")

# 取得示例文件（examples/）
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        print("⚠ 克隆仓库失败，拿不到 examples/ 示例文件；如果用的是自己上传的图，可以忽略。\n" + r.stderr[-500:])


def resolve(p):
    """相对路径依序在 /content 和仓库目录里找；绝对路径原样返回。"""
    if os.path.isabs(p):
        return p
    for base in ("/content", REPO_DIR):
        if os.path.exists(os.path.join(base, p)):
            return os.path.join(base, p)
    return os.path.join(REPO_DIR, p)


IMAGE_FILE = resolve(IMAGE_PATH)
if not os.path.exists(IMAGE_FILE):
    raise FileNotFoundError(f"找不到角色图 {IMAGE_FILE}：请确认 IMAGE_PATH，或先在左侧「文件」面板上传")
print("角色图：", IMAGE_FILE)

if ENABLE_VOICE_CLONE:
    if I_HAVE_CONSENT is not True:
        raise PermissionError("只能用你本人或已取得当事人同意的录音；确认后把 I_HAVE_CONSENT 改成 True")
    REF_FILE = resolve(REF_WAV)
    if not os.path.exists(REF_FILE):
        raise FileNotFoundError(f"找不到参考录音 {REF_FILE}：请先在左侧「文件」面板上传，并确认 REF_WAV")
    print("参考录音：", REF_FILE)
print("环境检查完成")

## ③ 配音（VoxCPM2）
- 默认：用 `VOICE` 选的**声音包**参考音（仓库 `voices/`）→ 每句用**可控克隆**生成（同一把声音，句首括号可加语气）。
- `VOICE` 选「自定义（用 VOICE_DESC）」：用 `VOICE_DESC` 做一次**音色设计**，存成参考音，之后同样逐句可控克隆。
- 开启声音克隆：用你的参考录音克隆。`REF_TEXT` 留空＝可控克隆（可加语气）；填了逐字稿＝极致克隆（更像本人，**不能**加语气括号）。
- 句子之间插入 `PAUSE_SEC` 秒停顿，接成一条音轨 `/content/output/voice.wav`。
- 同时记下每句在音轨里的开始／结束时间（字幕时间轴），存成 `/content/output/subtitle_timeline.json`。
- 跑完会删除模型并释放显存，再进入对嘴。生成耗时大约等于音频长度。
- ③-2 字幕准备：下载 ① 选的字体、检查字体缺不缺字。字体有问题会在这里先报错，不用等对嘴跑完才发现。

In [ ]:
# ③-1 配音：安装 VoxCPM2 → 音色设计（或用声音包／参考录音）→ 逐句生成 → 接成一条音轨（同时记下字幕时间轴）→ 释放显存
import gc, json, re

VOXCPM_PIP = "git+https://github.com/OpenBMB/VoxCPM.git@cce58a5b59303c9bd63d12f23afe6a49a4a80c59"  # PyPI 版的 generate() 没有 seed 参数，固定这个 commit
DESIGN_TEXT = "大家好，欢迎收看今天的节目，我们马上开始。"   # 音色设计时念的参考句（只用来定声音，不会出现在成品里）
MAX_CHARS = 80
TONE_RE = re.compile(r"^\s*[(（]([^()（）]*)[)）]\s*")


def split_long(text, limit=MAX_CHARS):
    """超过 limit 字时优先在标点处切开，找不到标点就硬切。"""
    parts = []
    while len(text) > limit:
        cut = max(text.rfind(p, 0, limit) for p in "。！？；，、!?;,")
        cut = cut + 1 if cut > 0 else limit
        parts.append(text[:cut].strip())
        text = text[cut:].strip()
    if text:
        parts.append(text)
    return parts


# 解析文稿：[(语气 或 None, 句子)]；句首括号的语气套用到该行切出来的每一段
LINES = []
for raw in SCRIPT.strip().splitlines():
    raw = raw.strip()
    if not raw:
        continue
    m = TONE_RE.match(raw)
    tone = (m.group(1).strip() or None) if m else None
    body = raw[m.end():] if m else raw
    for part in split_long(body):
        LINES.append((tone, part))
if not LINES:
    raise ValueError("SCRIPT 是空的：请在 ① 设置格里填写文稿")
EXTREME = ENABLE_VOICE_CLONE and REF_TEXT.strip() != ""
if EXTREME and any(t for t, _ in LINES):
    raise ValueError("极致克隆（填了 REF_TEXT）不能和语气括号一起用：请去掉句首括号，或把 REF_TEXT 留空改走可控克隆")
print(f"共 {len(LINES)} 句：")
for i, (t, s) in enumerate(LINES, 1):
    print(f"  {i:02d} {'(' + t + ')' if t else ''}{s}")

# 声音包：没开声音克隆、VOICE 也不是「自定义」时，用仓库 voices/ 里的参考音（先检查，免得装完模型才报错）
USE_PACK = not ENABLE_VOICE_CLONE and VOICE != "自定义（用 VOICE_DESC）"
if USE_PACK:
    pack_json = os.path.join(REPO_DIR, "voices", "voices.json")
    hit = []
    if os.path.exists(pack_json):
        with open(pack_json, encoding="utf-8") as f:
            hit = [v for v in json.load(f) if v["name"] == VOICE]
    PACK_WAV = os.path.join(REPO_DIR, "voices", hit[0]["file"]) if hit else ""
    if not PACK_WAV or not os.path.exists(PACK_WAV):
        raise FileNotFoundError(f"找不到声音包「{VOICE}」的参考音（应该在 {REPO_DIR}/voices/ 里）。"
                                "多半是 ② 克隆仓库失败：请重新运行 ② 再运行 ③，"
                                "或在 ① 把 VOICE 改成「自定义（用 VOICE_DESC）」")

t0 = time.time()
run([sys.executable, "-m", "pip", "install", "-q", VOXCPM_PIP])   # 会提示 gcsfs/fsspec 版本冲突，不影响
import numpy as np
import soundfile as sf
import torch
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
SR = model.tts_model.sample_rate
print(f"VoxCPM2 就绪（采样率 {SR}），安装＋载入 {time.time() - t0:.0f} 秒")


def synth(text, **kw):
    return model.generate(text=text, cfg_value=2.0, inference_timesteps=10, seed=SEED, **kw)


if ENABLE_VOICE_CLONE:
    ref_path = f"{OUT_DIR}/ref_clone.wav"
    run(["ffmpeg", "-v", "error", "-y", "-i", REF_FILE, "-ac", "1", ref_path])   # 统一转成单声道 wav
    clone_kw = dict(reference_wav_path=ref_path)
    if EXTREME:
        # 极致克隆：prompt_wav_path + prompt_text（逐字稿必须和录音一字不差）
        clone_kw.update(prompt_wav_path=ref_path, prompt_text=REF_TEXT.strip())
    print("声音来源：", "极致克隆" if EXTREME else "可控克隆", REF_FILE)
elif USE_PACK:
    clone_kw = dict(reference_wav_path=PACK_WAV)
    print(f"声音来源：声音包「{VOICE}」，参考音 {PACK_WAV}")
else:
    # 音色设计：描述写在 text 开头的括号里；每次结果不同，所以只做一次并存成参考音，之后都用克隆
    wav = synth(f"({VOICE_DESC}){DESIGN_TEXT}")
    ref_path = f"{OUT_DIR}/voice_design_ref.wav"
    sf.write(ref_path, wav, SR)
    clone_kw = dict(reference_wav_path=ref_path)
    print(f"音色设计完成：{ref_path}（{len(wav) / SR:.2f} 秒）")

pause = np.zeros(int(PAUSE_SEC * SR), dtype=np.float32)
pieces = []
SUB_TIMELINE = []   # 字幕时间轴：每句在音轨里的开始／结束秒数＋文稿原文（不含语气括号）
for i, (tone, text) in enumerate(LINES, 1):
    t1 = time.time()
    wav = np.asarray(synth(f"({tone}){text}" if tone else text, **clone_kw), dtype=np.float32).reshape(-1)
    if pieces:
        pieces.append(pause)
    start = sum(len(p) for p in pieces) / SR
    pieces.append(wav)
    SUB_TIMELINE.append({"start": round(start, 3), "end": round(start + len(wav) / SR, 3), "text": text})
    print(f"  {i:02d} 生成 {len(wav) / SR:.2f} 秒（耗时 {time.time() - t1:.0f} 秒）")
VOICE_WAV = f"{OUT_DIR}/voice.wav"
sf.write(VOICE_WAV, np.concatenate(pieces), SR)
SUB_TIMELINE_JSON = f"{OUT_DIR}/subtitle_timeline.json"   # 字幕关闭也照样存，之后可以用 ⑥ 补上字幕
with open(SUB_TIMELINE_JSON, "w", encoding="utf-8") as f:
    json.dump(SUB_TIMELINE, f, ensure_ascii=False, indent=1)
T_VOICE = time.time() - t0
print(f"配音完成：{VOICE_WAV}，共 {sum(len(p) for p in pieces) / SR:.2f} 秒；本步耗时 {T_VOICE:.0f} 秒")

# 释放显存，留给对嘴阶段
del model
gc.collect()
torch.cuda.empty_cache()
print("已释放 VoxCPM2 显存，目前显存占用：", gpu_used())

from IPython.display import Audio, display
display(Audio(VOICE_WAV))

In [ ]:
# ③-2 字幕准备：检查 ffmpeg、下载字体、检查字体缺不缺字（有问题在这里就报错，不用等对嘴跑完才发现）
#      本格同时定义 ⑤、⑥ 要用的字幕函数；真正把字幕烧进画面在 ⑤，换样式重做用 ⑥
import json, math, os, re, shutil, time, unicodedata, urllib.request

SUB_TIMELINE_JSON = f"{OUT_DIR}/subtitle_timeline.json"   # ③-1 记下的字幕时间轴
CLEAN_MP4 = f"{OUT_DIR}/talking_avatar_nosub.mp4"         # ⑤ 合成的不带字幕版本（⑥ 拿它重烧）
OUT_MP4 = f"{OUT_DIR}/talking_avatar.mp4"                 # 成品（字幕开启时带字幕）
SUB_ASS = f"{OUT_DIR}/talking_avatar.ass"                 # 烧进画面用的字幕：ASS 格式，样式和效果都写在里面
SUB_SRT = f"{OUT_DIR}/talking_avatar.srt"                 # 另外给你的纯文字字幕（可导入剪辑软件或上传到平台）
FONT_DIR = "/content/fonts"
CUSTOM_FONT = "自定义（上传字体文件）"

# 可选字体：全部是开源、可商用的字体（SIL Open Font License 1.1），只从官方发布处下载
#   名称: (下载网址, 发布处)
SUB_FONTS = {
    "思源黑体（粗）": ("https://github.com/adobe-fonts/source-han-sans/raw/release/OTF/SimplifiedChinese/SourceHanSansSC-Bold.otf",
                "Adobe 官方 GitHub adobe-fonts/source-han-sans"),
    "霞鹜文楷": ("https://github.com/lxgw/LxgwWenKai/releases/latest/download/LXGWWenKai-Regular.ttf",
             "作者官方 GitHub Releases lxgw/LxgwWenKai"),
    "站酷快乐体": ("https://github.com/google/fonts/raw/main/ofl/zcoolkuaile/ZCOOLKuaiLe-Regular.ttf",
              "Google Fonts 官方 GitHub google/fonts"),
    "站酷小薇LOGO体": ("https://github.com/google/fonts/raw/main/ofl/zcoolxiaowei/ZCOOLXiaoWei-Regular.ttf",
                  "Google Fonts 官方 GitHub google/fonts"),
    "站酷庆科黄油体": ("https://github.com/google/fonts/raw/main/ofl/zcoolqingkehuangyou/ZCOOLQingKeHuangYou-Regular.ttf",
                "Google Fonts 官方 GitHub google/fonts"),
}
FONT_MAGIC = (b"\x00\x01\x00\x00", b"OTTO", b"true", b"ttcf")   # .ttf／.otf／.ttc 文件开头的 4 个字节
ALIGN = {"底部": 2, "中间": 5, "顶部": 8}                        # ASS 对齐代号：底部居中／正中间／顶部居中
EFFECTS = ("无", "淡入", "淡入淡出", "弹出", "滑入", "逐字")
BREAK_AFTER = "，。！？；：、,.!?;:）)」』”》…"                  # 一行放不下时，优先在这些标点后面切
NO_LINE_START = "，。！？；：、,.!?;:）)」』”’》…—"             # 这些标点不放在一行的开头


def check_ffmpeg():
    """烧字幕要 ffmpeg 的 ass 字幕滤镜（libass）和 libx264 编码器。"""
    if not re.search(r"^\s*\S+\s+ass\s", run(["ffmpeg", "-hide_banner", "-filters"]).stdout, re.M):
        raise RuntimeError("这台机器的 ffmpeg 没有 ass 字幕滤镜（libass），烧不了字幕。"
                           "可以在 ① 把 SUBTITLE 改成 False，只出不带字幕的成品")
    if "libx264" not in run(["ffmpeg", "-hide_banner", "-encoders"]).stdout:
        raise RuntimeError("这台机器的 ffmpeg 没有 libx264 编码器，烧不了字幕。"
                           "可以在 ① 把 SUBTITLE 改成 False，只出不带字幕的成品")


def ass_color(rgb, name):
    """#RRGGBB → ASS 的 &H00BBGGRR（ASS 颜色是蓝绿红倒过来写，前面的 00＝不透明）。"""
    h = str(rgb).strip().lstrip("#")
    if not re.fullmatch(r"[0-9A-Fa-f]{6}", h):
        raise ValueError(f"{name} 要写成 #RRGGBB，例如白色 #FFFFFF；现在是 {rgb!r}")
    return "&H00" + (h[4:6] + h[2:4] + h[0:2]).upper()


def _is_font(path):
    with open(path, "rb") as f:
        return f.read(4) in FONT_MAGIC


def prepare_font():
    """准备 ① 选的字体；回传 (字体文件夹, 字体文件)。每种字体单独一个文件夹，烧字幕时只让 ffmpeg 看到这一个。"""
    if SUB_FONT == CUSTOM_FONT:
        src = resolve(SUB_FONT_FILE)
        if not os.path.exists(src):
            raise FileNotFoundError(f"找不到自定义字体 {src}：请先上传到左侧「文件」面板，再确认 ① 的 SUB_FONT_FILE")
        if not _is_font(src):
            raise ValueError(f"{src} 不是 .ttf／.otf／.ttc 字体文件")
        d = f"{FONT_DIR}/custom"
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d)
        return d, shutil.copy(src, d)
    if SUB_FONT not in SUB_FONTS:
        raise ValueError(f"不认识的字体 {SUB_FONT!r}：请在 ① 的 SUB_FONT 下拉选单里选")
    url, origin = SUB_FONTS[SUB_FONT]
    name = os.path.basename(url)
    d = f"{FONT_DIR}/{os.path.splitext(name)[0]}"
    path = f"{d}/{name}"
    if not os.path.exists(path):
        os.makedirs(d, exist_ok=True)
        print(f"下载字体「{SUB_FONT}」（SIL OFL 1.1，来源：{origin}）……")
        try:
            with urllib.request.urlopen(url, timeout=120) as r, open(path + ".part", "wb") as f:
                shutil.copyfileobj(r, f)
        except Exception as e:
            raise RuntimeError(f"字体「{SUB_FONT}」下载失败：{e}\n网址：{url}\n"
                               f"可以在 ① 换一个字体，或自己下载字体文件后选「{CUSTOM_FONT}」") from None
        if not _is_font(path + ".part"):
            os.remove(path + ".part")
            raise RuntimeError(f"下载到的不是字体文件（网址可能已经失效）：{url}\n"
                               f"可以在 ① 换一个字体，或自己下载字体文件后选「{CUSTOM_FONT}」")
        os.replace(path + ".part", path)
        print(f"  下载完成，{os.path.getsize(path) / 1e6:.1f}MB")
    return d, path


def font_meta(path):
    """读字体内部的家族名（ASS 靠它找字体）、是否粗体、收录了哪些字（None＝没办法检查）。"""
    chars = None
    try:
        from fontTools.ttLib import TTFont
        ft = TTFont(path, fontNumber=0, lazy=True)
        family = ft["name"].getDebugName(1)
        bold = "OS/2" in ft and ft["OS/2"].usWeightClass >= 600
        chars = set(ft.getBestCmap() or {})
        ft.close()
        if family:
            return family, bold, chars
    except ImportError:
        pass
    from PIL import ImageFont   # 没有 fontTools、或读不到家族名时，改用 PIL 读
    family, style = ImageFont.truetype(path, 20).getname()
    return family, "bold" in (style or "").lower(), chars


def load_timeline():
    if not os.path.exists(SUB_TIMELINE_JSON):
        raise FileNotFoundError(f"找不到字幕时间轴 {SUB_TIMELINE_JSON}：要先跑完 ③-1 配音。"
                                "删除过运行时的话文件已经没了，只能重新全部运行")
    with open(SUB_TIMELINE_JSON, encoding="utf-8") as f:
        return json.load(f)


def prepare_subtitles():
    """检查设置和 ffmpeg、准备字体、检查缺字；回传 (时间轴, 字体文件夹, 字体家族名, 是否粗体)。"""
    if SUB_EFFECT not in EFFECTS:
        raise ValueError(f"SUB_EFFECT 只能是 {'、'.join(EFFECTS)}；现在是 {SUB_EFFECT!r}")
    if SUB_POSITION not in ALIGN:
        raise ValueError(f"SUB_POSITION 只能是 底部、中间、顶部；现在是 {SUB_POSITION!r}")
    ass_color(SUB_COLOR, "SUB_COLOR")
    ass_color(SUB_OUTLINE_COLOR, "SUB_OUTLINE_COLOR")
    check_ffmpeg()
    timeline = load_timeline()
    font_dir, font_path = prepare_font()
    family, bold, chars = font_meta(font_path)
    if chars is None:
        print("（没有 fontTools，跳过缺字检查）")
    else:
        miss = sorted({c for seg in timeline for c in seg["text"] if not c.isspace() and ord(c) not in chars})
        if miss:
            print(f"⚠ 字体「{SUB_FONT}」缺少这些字：{''.join(miss)}\n"
                  "  烧进画面时这些字会被换成别的字体，或显示成方框；介意的话在 ① 换字体（思源黑体收的字最全）")
    return timeline, font_dir, family, bold


def _char_w(c):
    """估算字宽（以字号为 1）：中文等全角字算 1，英文、数字等半角字约 0.55。"""
    return 1.0 if unicodedata.east_asian_width(c) in "WFA" else 0.55


def _spoken(s):
    """念出来的字数（不算空格和标点）；切段后按它分配时间。"""
    return sum(1 for c in s if not c.isspace() and not unicodedata.category(c).startswith("P"))


def wrap_text(text, max_w):
    """一行放不下时切成几段：尽量等长、优先在标点后面切、不拆开英文单词、标点不放行首。"""
    parts, rest = [], text.strip()
    while sum(map(_char_w, rest)) > max_w:
        total = sum(map(_char_w, rest))
        target = total / math.ceil(total / max_w)
        best, best_score, acc = 1, None, 0.0
        for i in range(1, len(rest)):
            acc += _char_w(rest[i - 1])
            if acc > max_w:
                break
            score = abs(acc - target)
            if rest[i - 1] in BREAK_AFTER:
                score -= max_w * 0.3
            if rest[i - 1].isascii() and rest[i - 1].isalnum() and rest[i].isascii() and rest[i].isalnum():
                score += max_w
            if rest[i] in NO_LINE_START:
                score += max_w * 2
            if best_score is None or score < best_score:
                best, best_score = i, score
        parts.append(rest[:best].strip())
        rest = rest[best:].strip()
    if rest:
        parts.append(rest)
    return parts


def build_events(timeline, max_w):
    """每句切成一行一段，按字数比例分时间；回传 [显示开始, 显示结束, 这段念完的时间, 文字]（秒）。"""
    events = []
    for seg in timeline:
        parts = wrap_text(seg["text"], max_w)
        weights = [max(1, _spoken(p)) for p in parts]
        t, dur = seg["start"], seg["end"] - seg["start"]
        for p, w in zip(parts, weights):
            t_end = t + dur * w / sum(weights)
            events.append([t, t_end, t_end, p])
            t = t_end
    # 句间停顿时让上一段字幕继续留着，直到下一段出现（避免字幕一闪一闪）
    for a, b in zip(events, events[1:]):
        a[1] = b[0]
    return events


def _ass_t(s):
    cs = max(0, round(s * 100))
    return f"{cs // 360000}:{cs // 6000 % 60:02d}:{cs // 100 % 60:02d}.{cs % 100:02d}"


def _srt_t(s):
    ms = max(0, round(s * 1000))
    return f"{ms // 3600000:02d}:{ms // 60000 % 60:02d}:{ms // 1000 % 60:02d},{ms % 1000:03d}"


def _esc(s):
    """ASS 里大括号和反斜线是指令符号，换成全角，免得字幕文字被当成指令。"""
    return s.replace("\\", "＼").replace("{", "｛").replace("}", "｝")


def effect_text(text, show_ms, speech_ms, x, y, font_px):
    """按 SUB_EFFECT 给这一段字幕加 ASS 特效标签。show_ms＝显示多久；speech_ms＝这段念多久（逐字用）；x, y＝定位点（滑入用）。"""
    t = _esc(text)
    fade = max(1, min(200, show_ms // 3))
    if SUB_EFFECT == "淡入":
        return rf"{{\fad({fade},0)}}" + t
    if SUB_EFFECT == "淡入淡出":
        return rf"{{\fad({fade},{fade})}}" + t
    if SUB_EFFECT == "弹出":   # 从 60% 放大到 110%，再回到 100%
        p1, p2 = max(1, min(120, show_ms // 4)), max(2, min(200, show_ms // 3))
        return rf"{{\fscx60\fscy60\t(0,{p1},\fscx110\fscy110)\t({p1},{p2},\fscx100\fscy100)}}" + t
    if SUB_EFFECT == "滑入":   # 从下方 0.6 个字高的地方往上滑到定位点，同时淡入
        s, dy = max(1, min(250, show_ms // 3)), round(font_px * 0.6)
        return rf"{{\move({x},{y + dy},{x},{y},0,{s})\fad({s},0)}}" + t
    if SUB_EFFECT == "逐字":   # 每个字先全透明，念到它的时间点在 80 毫秒内变成不透明；标点、空格跟着前一个字出现
        n, k, out = max(1, _spoken(text)), 0, []
        for c in text:
            if c.isspace() or unicodedata.category(c).startswith("P"):
                t0 = round(speech_ms * max(0, k - 1) / n)
            else:
                t0 = round(speech_ms * k / n)
                k += 1
            out.append(rf"{{\alpha&HFF&\t({t0},{t0 + 80},\alpha&H00&)}}" + _esc(c))
        return "".join(out)
    return t   # 无


def write_subtitles(timeline, family, bold, W, H):
    """按画面尺寸 W×H 生成 ASS（烧进画面用）和 SRT（另外给的）；回传字幕段数。"""
    font_px = max(8, round(min(W, H) / 12 * SUB_SIZE / 100))   # 字号：画面短边的 1/12 × SUB_SIZE%
    outline_px = round(font_px * SUB_OUTLINE / 100, 1)          # 描边：字号的 SUB_OUTLINE%
    margin_v = round(H * SUB_MARGIN / 100)                      # 上下边距：画面高度的 SUB_MARGIN%
    margin_lr = max(8, round(W * 0.06))                         # 左右各留画面宽度的 6%
    max_w = (W - 2 * margin_lr - 2 * outline_px) / font_px      # 一行最多放几个中文字宽
    events = build_events(timeline, max_w)
    align = ALIGN[SUB_POSITION]
    x, y = W // 2, {2: H - margin_v, 5: H // 2, 8: margin_v}[align]   # 滑入要用的定位点
    ass = [
        "[Script Info]", "ScriptType: v4.00+", f"PlayResX: {W}", f"PlayResY: {H}",
        "WrapStyle: 2",   # 不自动换行：长句已经切成一行一段
        "ScaledBorderAndShadow: yes", "YCbCr Matrix: None", "",
        "[V4+ Styles]",
        "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, "
        "Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, "
        "MarginL, MarginR, MarginV, Encoding",
        f"Style: Default,{family},{font_px},{ass_color(SUB_COLOR, 'SUB_COLOR')},&H000000FF,"
        f"{ass_color(SUB_OUTLINE_COLOR, 'SUB_OUTLINE_COLOR')},&H00000000,{-1 if bold else 0},0,0,0,"
        f"100,100,0,0,1,{outline_px},0,{align},{margin_lr},{margin_lr},{margin_v},1",
        "",
        "[Events]",
        "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text",
    ]
    srt = []
    for i, (s, e, se, text) in enumerate(events, 1):
        show_ms, speech_ms = round((e - s) * 1000), round((se - s) * 1000)
        ass.append(f"Dialogue: 0,{_ass_t(s)},{_ass_t(e)},Default,,0,0,0,,"
                   f"{effect_text(text, show_ms, speech_ms, x, y, font_px)}")
        srt.append(f"{i}\n{_srt_t(s)} --> {_srt_t(e)}\n{text}\n")
    with open(SUB_ASS, "w", encoding="utf-8") as f:
        f.write("\n".join(ass) + "\n")
    with open(SUB_SRT, "w", encoding="utf-8") as f:
        f.write("\n".join(srt))
    print(f"  画面 {W}x{H}：字号 {font_px} 像素、描边 {outline_px} 像素，一行最多约 {max_w:.1f} 个中文字")
    return len(events)


def video_size():
    """画面尺寸用 notebook 的 WIDTH、HEIGHT；重启过会话（变量没了）时，从不带字幕的版本读回来。"""
    if "WIDTH" in globals() and "HEIGHT" in globals():
        return WIDTH, HEIGHT
    wh = run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=width,height",
              "-of", "csv=p=0:s=x", CLEAN_MP4]).stdout.strip()
    print(f"（没有 WIDTH／HEIGHT 变量，多半是重启过会话：画面尺寸从 {CLEAN_MP4} 读到 {wh}）")
    w, h = wh.split("x")
    return int(w), int(h)


def sub_signature():
    """现在的字幕设置＋素材文件的修改时间；和上次做成品时一样，⑥ 就不重做。"""
    paths = [CLEAN_MP4, SUB_TIMELINE_JSON] + ([resolve(SUB_FONT_FILE)] if SUB_FONT == CUSTOM_FONT else [])
    return (SUBTITLE, SUB_FONT, SUB_FONT_FILE, SUB_SIZE, SUB_EFFECT, SUB_POSITION, SUB_MARGIN, SUB_OUTLINE,
            SUB_COLOR, SUB_OUTLINE_COLOR, tuple(os.path.getmtime(p) if os.path.exists(p) else 0 for p in paths))


def make_final_video():
    """按 ① 现在的字幕设置，从不带字幕的版本做出成品 OUT_MP4；回传要下载的文件。⑤ 和 ⑥ 都用它。"""
    global SUB_LAST_SIG
    if not os.path.exists(CLEAN_MP4):
        raise FileNotFoundError(f"找不到不带字幕的版本 {CLEAN_MP4}：要先完整跑完一次（到 ⑤）。"
                                "删除过运行时的话文件已经没了，只能重新全部运行")
    if not SUBTITLE:
        shutil.copyfile(CLEAN_MP4, OUT_MP4)
        SUB_LAST_SIG = sub_signature()
        print("字幕：关闭（① 的 SUBTITLE = False），成品不带字幕")
        return [OUT_MP4]
    timeline, font_dir, family, bold = prepare_subtitles()
    W, H = video_size()
    n = write_subtitles(timeline, family, bold, W, H)
    t0 = time.time()
    try:
        r = run(["ffmpeg", "-hide_banner", "-nostats", "-y", "-i", CLEAN_MP4,
                 "-vf", f"ass={SUB_ASS}:fontsdir={font_dir}",
                 "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p",
                 "-c:a", "copy", "-movflags", "+faststart", OUT_MP4])
    except RuntimeError as e:
        raise RuntimeError(f"{e}\n\n烧字幕失败。不带字幕的版本在 {CLEAN_MP4}（左侧「文件」面板 output 里可以下载）；"
                           "改好 ① 的字幕设置后，运行 ① 再运行 ⑥ 只重做字幕") from None
    used = sorted({ln.split("->", 1)[1].strip() for ln in r.stderr.splitlines() if "fontselect:" in ln and "->" in ln})
    print(f"字幕已烧进画面：{n} 段，字体「{SUB_FONT}」（字体内部名称 {family}），效果「{SUB_EFFECT}」，"
          f"耗时 {time.time() - t0:.0f} 秒")
    if used:
        print("  ffmpeg 实际用的字体：" + "；".join(used))
    SUB_LAST_SIG = sub_signature()
    return [OUT_MP4, SUB_SRT]


if SUBTITLE:
    _tl, _dir, _family, _bold = prepare_subtitles()
    print(f"字幕准备完成：{len(_tl)} 句，字体「{SUB_FONT}」（字体内部名称 {_family}），效果「{SUB_EFFECT}」")
else:
    print("字幕：关闭（① 的 SUBTITLE = False），⑤ 会输出不带字幕的成品")

## ④ 对嘴（ComfyUI + InfiniteTalk）
- ComfyUI 固定 **v0.43.0**；模型 6 个文件全是 `.safetensors`，合计约 30GB，每次开机从 HuggingFace 下载（commit 已固定）。
- 画面 480×832（竖屏），6 步；角色图先裁成 480:832 比例（保留上方、裁掉下方）再缩放。
- 每段 81 帧（25fps，约 3.24 秒）；配音更长时一段接一段延伸（每段和前面重叠 9 帧当动作参考），最后裁回配音长度。
- 作者实测：21.5 秒的成品，这一步约 36 分钟。每 30 秒会打印一次进度。

In [ ]:
# ④-1 安装 ComfyUI（固定版本 v0.43.0）
COMFY = "/content/ComfyUI"
COMFY_TAG = "v0.43.0"   # 节点输入名称按这个版本的源码写，换版本可能要改节点图
t0 = time.time()
if not os.path.isdir(COMFY):
    run(["git", "clone", "--depth", "1", "--branch", COMFY_TAG, "https://github.com/comfyanonymous/ComfyUI", COMFY])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=COMFY)
T_INSTALL = time.time() - t0
print(f"ComfyUI {COMFY_TAG} 安装完成，{T_INSTALL:.0f} 秒")

In [ ]:
# ④-2 下载模型（只下载 .safetensors，约 30GB；来源与文件名照 ComfyUI 官方 InfiniteTalk 教程）
import shutil
from huggingface_hub import hf_hub_download

# (仓库, 文件路径, 固定的 commit, ComfyUI models 子目录)
MODELS = [
    ("Kijai/WanVideo_comfy_fp8_scaled", "I2V/Wan2_1-I2V-14B-480p_fp8_e4m3fn_scaled_KJ.safetensors",
     "033a4e487f60220b3d6e469599a6aebc46e13cee", "diffusion_models"),   # 16.6GB 底模 Wan2.1 I2V 14B 480P fp8
    ("Comfy-Org/Wan-2.1", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     "123acf1cc74bccbb9bfff8ac1ee72edc08c2341d", "text_encoders"),      # 6.7GB 文本编码器
    ("Comfy-Org/Wan-2.1", "split_files/model_patches/wan2.1_infiniteTalk_single_fp16.safetensors",
     "123acf1cc74bccbb9bfff8ac1ee72edc08c2341d", "model_patches"),      # 5.1GB InfiniteTalk 单人
    ("Kijai/wav2vec2_safetensors", "wav2vec2-chinese-base_fp16.safetensors",
     "87847d3bc53702afda44078249e7c33e867827c4", "audio_encoders"),     # 0.19GB 中文 wav2vec2
    ("Kijai/WanVideo_comfy", "Wan2_1_VAE_bf16.safetensors",
     "8260d429d19fd7a72304cad059160b95d843913f", "vae"),                # 0.25GB
    ("Kijai/WanVideo_comfy", "Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors",
     "8260d429d19fd7a72304cad059160b95d843913f", "loras"),              # 0.74GB 加速 LoRA
]
print(f"磁盘剩余 {shutil.disk_usage('/content').free / 1e9:.0f}GB（需要约 30GB）")
t0 = time.time()
for repo, path, rev, folder in MODELS:
    name = os.path.basename(path)
    assert name.endswith(".safetensors"), name
    dst = f"{COMFY}/models/{folder}/{name}"
    if os.path.exists(dst):
        continue
    print("下载：", name)
    tmp = hf_hub_download(repo, path, revision=rev, local_dir="/content/hf_tmp")
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.move(tmp, dst)
T_DOWNLOAD = time.time() - t0
print(f"模型就绪，{T_DOWNLOAD:.0f} 秒")

In [ ]:
# ④-3 准备输入：角色图裁成 480:832 并缩放；配音转成 16kHz 单声道给对嘴模型用
import math
from PIL import Image

WIDTH, HEIGHT = 480, 832   # 480P 竖屏；必须是 16 的倍数
STEPS = 6                  # 搭配 lightx2v 加速 LoRA，官方模板用 6 步
if not os.path.exists(VOICE_WAV):
    raise FileNotFoundError(f"找不到配音 {VOICE_WAV}：请先运行 ③")

IMG_NAME, AUD_NAME = "ta_host.png", "ta_voice.wav"
im = Image.open(IMAGE_FILE).convert("RGB")
w0, h0 = im.size
target = WIDTH / HEIGHT
if w0 / h0 > target:   # 太宽：左右各裁一半
    cw = round(h0 * target)
    box = ((w0 - cw) // 2, 0, (w0 - cw) // 2 + cw, h0)
else:                  # 太高：保留上方（头部），裁掉下方
    box = (0, 0, w0, round(w0 / target))
im.crop(box).resize((WIDTH, HEIGHT), Image.LANCZOS).save(f"{COMFY}/input/{IMG_NAME}")
print(f"图：{w0}x{h0} → 裁切 {box} → {WIDTH}x{HEIGHT}")

run(["ffmpeg", "-v", "error", "-y", "-i", VOICE_WAV, "-ac", "1", "-ar", "16000", f"{COMFY}/input/{AUD_NAME}"])
AUDIO_SEC = float(run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0",
                       f"{COMFY}/input/{AUD_NAME}"]).stdout)

# 每段 81 帧；超过就接延伸段（重叠 9 帧当动作参考），最后裁回配音长度
FPS, CHUNK, MOTION = 25, 81, 9
NEED_FRAMES = math.ceil(AUDIO_SEC * FPS)
N_CHUNKS = 1 + max(0, math.ceil((NEED_FRAMES - CHUNK) / (CHUNK - MOTION)))
print(f"配音 {AUDIO_SEC:.2f} 秒 → {NEED_FRAMES} 帧，分 {N_CHUNKS} 段生成")
if AUDIO_SEC > 22:
    print("⚠ 提示：作者实测过的最长成品是 21.5 秒（8 段，约 36 分钟）；更长时耗时按段数增加，系统内存够不够未验证")

In [ ]:
# ④-4 在后台启动 ComfyUI（只监听 127.0.0.1，不开网页界面）
import json, threading, urllib.request, urllib.error

API = "http://127.0.0.1:8188"
args = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"]
if LOW_VRAM:
    args.append("--novram")


def _serve():
    # 在后台线程里跑 ComfyUI，输出写进 /content/comfyui.log
    with open("/content/comfyui.log", "w") as log:
        subprocess.run(args, cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)


def _post(path, payload):
    req = urllib.request.Request(API + path, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        return json.load(urllib.request.urlopen(req))
    except urllib.error.HTTPError as e:
        raise RuntimeError(e.read().decode()) from None


def _get(path):
    return json.load(urllib.request.urlopen(API + path))


# 重跑时先关掉上一次留下的 ComfyUI（重启会话后它可能还占着显存和端口）
subprocess.run(["pkill", "-f", "main.py --listen 127.0.0.1 --port 8188"])
time.sleep(3)
t0 = time.time()
server = threading.Thread(target=_serve, daemon=True)
server.start()
for _ in range(300):
    if not server.is_alive():
        raise RuntimeError("ComfyUI 启动失败，请看 /content/comfyui.log")
    try:
        _get("/system_stats")
        break
    except (urllib.error.URLError, ConnectionError):
        time.sleep(1)
else:
    raise RuntimeError("等了 300 秒还没启动，请看 /content/comfyui.log")
T_BOOT = time.time() - t0
if "WanInfiniteTalkToVideo" not in _get("/object_info/WanInfiniteTalkToVideo"):
    raise RuntimeError("这个版本的 ComfyUI 没有 WanInfiniteTalkToVideo 节点")
print(f"ComfyUI 已启动，{T_BOOT:.0f} 秒；LOW_VRAM={LOW_VRAM}")

In [ ]:
# ④-5 生成对嘴视频（每 30 秒打印一次进度）
M = {folder: os.path.basename(path) for _, path, _, folder in MODELS}


def build_graph():
    # 节点与参数照官方模板 video_wan2_1_infinitetalk.json（单人）：lightx2v LoRA、euler、normal、cfg 1、audio_scale 1
    g = {}

    def add(cls, **inputs):
        nid = str(len(g) + 1)
        g[nid] = {"class_type": cls, "inputs": inputs}
        return nid

    unet = add("UNETLoader", unet_name=M["diffusion_models"], weight_dtype="default")
    model = add("LoraLoaderModelOnly", model=[unet, 0], lora_name=M["loras"], strength_model=1.0)
    clip = add("CLIPLoader", clip_name=M["text_encoders"], type="wan", device="default")
    vae = add("VAELoader", vae_name=M["vae"])
    patch = add("ModelPatchLoader", name=M["model_patches"])
    aenc = add("AudioEncoderLoader", audio_encoder_name=M["audio_encoders"])
    img = add("LoadImage", image=IMG_NAME)
    aud = add("LoadAudio", audio=AUD_NAME)
    aemb = add("AudioEncoderEncode", audio_encoder=[aenc, 0], audio=[aud, 0])
    pos = add("CLIPTextEncode", clip=[clip, 0], text=PROMPT)
    neg = add("ConditioningZeroOut", conditioning=[pos, 0])
    sampler = add("KSamplerSelect", sampler_name="euler")
    frames = None
    for i in range(N_CHUNKS):
        it_in = dict(mode="single_speaker", model=[model, 0], model_patch=[patch, 0], positive=[pos, 0],
                     negative=[neg, 0], vae=[vae, 0], width=WIDTH, height=HEIGHT, length=CHUNK,
                     start_image=[img, 0], audio_encoder_output_1=[aemb, 0], motion_frame_count=MOTION,
                     audio_scale=1.0)
        if frames is not None:
            it_in["previous_frames"] = [frames, 0]   # 延伸段：接在前面所有帧之后
        it = add("WanInfiniteTalkToVideo", **it_in)
        guider = add("CFGGuider", model=[it, 0], positive=[it, 1], negative=[it, 2], cfg=1.0)
        sigmas = add("BasicScheduler", model=[it, 0], scheduler="normal", steps=STEPS, denoise=1.0)
        noise = add("RandomNoise", noise_seed=SEED + i)
        lat = add("SamplerCustomAdvanced", noise=[noise, 0], guider=[guider, 0], sampler=[sampler, 0],
                  sigmas=[sigmas, 0], latent_image=[it, 3])
        dec = add("VAEDecode", samples=[lat, 0], vae=[vae, 0])
        if frames is None:
            frames = dec
        else:
            new = add("ImageFromBatch", image=[dec, 0], batch_index=MOTION, length=4096)   # 去掉重叠的动作参考帧
            frames = add("ImageBatch", image1=[frames, 0], image2=[new, 0])
    final = add("ImageFromBatch", image=[frames, 0], batch_index=0, length=NEED_FRAMES)
    video = add("CreateVideo", images=[final, 0], fps=float(FPS), audio=[aud, 0])
    save = add("SaveVideo", video=[video, 0], filename_prefix="talking_avatar/raw",
               format="auto", **{"format.codec": "auto"})
    return g, save


graph, save_id = build_graph()
t0 = time.time()
pid = _post("/prompt", {"prompt": graph})["prompt_id"]
last = 0
while True:
    hist = _get(f"/history/{pid}")
    if pid in hist:
        break
    if not server.is_alive():
        raise RuntimeError("ComfyUI 进程结束了（多半是系统内存或显存不够），请看 /content/comfyui.log")
    if time.time() - last >= 30:
        last = time.time()
        with open("/content/comfyui.log", "rb") as f:
            f.seek(max(0, os.path.getsize("/content/comfyui.log") - 400))
            tail = f.read().decode(errors="ignore").replace("\r", "\n").strip().splitlines()
        print(f"[{time.time() - t0:5.0f} 秒] {tail[-1][:150] if tail else ''}")
    time.sleep(2)
T_GEN = time.time() - t0

status = hist[pid].get("status", {})
if status.get("status_str") == "error":
    raise RuntimeError(json.dumps(status.get("messages"), ensure_ascii=False)[:3000])
v = hist[pid]["outputs"][save_id]["images"][0]
RAW_MP4 = os.path.join(COMFY, "output", v["subfolder"], v["filename"])
print(f"对嘴完成：{RAW_MP4}，生成耗时 {T_GEN / 60:.1f} 分钟")

In [ ]:
# ⑤ 合成输出：画面原样保留，音轨换回原始配音（ComfyUI 输出里的是转过的 16kHz 版）；字幕开启时再把字幕烧进画面
from IPython.display import Video, display
from google.colab import files

# 先合成不带字幕的版本 CLEAN_MP4（⑥ 换字幕样式时拿它重烧），再按 ① 的字幕设置做出成品 OUT_MP4（路径定义在 ③-2）
run(["ffmpeg", "-v", "error", "-y", "-i", RAW_MP4, "-i", VOICE_WAV, "-map", "0:v", "-map", "1:a",
     "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart", CLEAN_MP4])
DOWNLOADS = make_final_video()

print("=" * 60)
print(f"GPU：{GPU_NAME}")
print(f"成品：{OUT_MP4}（{WIDTH}x{HEIGHT}，配音 {AUDIO_SEC:.2f} 秒，{N_CHUNKS} 段）")
if SUBTITLE:
    print(f"字幕：已烧进画面，另附 {SUB_SRT}；想换字幕样式用 ⑥（不用重跑配音和对嘴）")
print(f"耗时：配音 {T_VOICE:.0f} 秒、装 ComfyUI {T_INSTALL:.0f} 秒、下载模型 {T_DOWNLOAD:.0f} 秒、"
      f"启动 {T_BOOT:.0f} 秒、对嘴 {T_GEN:.0f} 秒；从 ② 到现在共 {(time.time() - T_START) / 60:.1f} 分钟")
print("浏览器会自动下载；没弹出来就从左侧「文件」面板的 output 文件夹下载")
print("=" * 60)
for _f in DOWNLOADS:
    files.download(_f)
display(Video(OUT_MP4, embed=True, width=360))

## ⑥ 只重做字幕（换字幕样式时用）
跑完一次后，想换字幕的字体、字号、效果、位置、颜色，或打开／关闭字幕：
1. 改 ① 设置格里「字幕」那一段。
2. 运行 ① 设置格（点格子左边的 ▶）。
3. 运行下面这一格，会自动下载新的 `talking_avatar.mp4`（和 `talking_avatar.srt`）。

不会重跑配音和对嘴：拿 `/content/output/` 里不带字幕的版本 `talking_avatar_nosub.mp4` 和字幕时间轴 `subtitle_timeline.json` 重新烧字幕。
- 「全部运行」时这一格会自动跳过（字幕设置没变）。
- 中途「重启会话」过：先依次运行 ①、②、③-2（③-2 只准备字体，很快），再运行这一格。**不要**运行 ③-1，会重新配音。
- **删除运行时后文件就没了，不能再重做字幕**，所以换样式要在删除运行时之前做。
- 字幕文字固定用文稿原文；想改字幕文字，要改 ① 的文稿重新全部运行。

In [ ]:
# ⑥ 只重做字幕：用 ① 现在的字幕设置，拿不带字幕的版本重烧（不重跑配音和对嘴）
import os
from IPython.display import Video, display
from google.colab import files

if any(n not in globals() for n in ("run", "resolve", "make_final_video")):
    raise RuntimeError("缺少前面格子定义的函数（多半是重启过会话）：请先依次运行 ①、②、③-2，再运行本格。"
                       "不要运行 ③-1（会重新配音）")
if not os.path.exists(CLEAN_MP4):
    raise FileNotFoundError(f"找不到不带字幕的版本 {CLEAN_MP4}：要先完整跑完一次（到 ⑤）。"
                            "删除过运行时的话文件已经没了，只能重新全部运行")
if globals().get("SUB_LAST_SIG") == sub_signature():
    print("字幕设置和现在的成品一样，跳过（「全部运行」时这一格本来就会跳过）。\n"
          "要换样式：改 ① 的字幕设置 → 运行 ① → 再运行本格")
else:
    for _f in make_final_video():
        files.download(_f)
    display(Video(OUT_MP4, embed=True, width=360))

## 跑完之后（重要）
- 确认浏览器下载文件夹里有 `talking_avatar.mp4`（开字幕时还有 `talking_avatar.srt`；浏览器问是否允许下载多个文件时点允许）；没有就从左侧「文件」面板 `output/` 里下载。
- 想换字幕样式：要在删除运行时**之前**用 ⑥ 只重做字幕。
- **「代码执行程序 → 断开连接并删除运行时」**（英文界面：Runtime → Disconnect and delete runtime）。**闲置也会扣计算单元。** 第一次点可能先弹出警告，确认后要再点一次删除。
- 爆显存（报错里有 `OutOfMemory` / `OOM`）：「代码执行程序 → 重启会话」→ ① 改 `LOW_VRAM = True` → 全部运行（已装的 ComfyUI 和已下载的模型会跳过）。